# CODLING Evaluation on HumanEval
=================================

This notebook evaluates CODLING on the HumanEval benchmark for code generation.

**Contents:**
- Load trained model
- Run code generation tests
- Measure pass@k metrics
- Compare with baselines
- Demo: generate Python/TypeScript code

---

## 1. Setup and Imports

In [ ]:
import sys
sys.path.insert(0, '/content')
import os
import torch
import torch.nn as nn
from transformers import AutoTokenizer
import numpy as np
from tqdm import tqdm
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
CHECKPOINT_DIR = '/content/drive/MyDrive/codling_checkpoints'
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

## 2. Load Tokenizer

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("gpt2")
special_tokens = {"pad_token": "<|pad|>", "bos_token": "<|bos|>", "eos_token": "<|eos|>"}
tokenizer.add_special_tokens(special_tokens)
print(f"Tokenizer loaded: {len(tokenizer)} tokens")

## 3. Load Model

In [ ]:
import torch.nn.functional as F

class SimplifiedSSMBlock(nn.Module):
    def __init__(self, hidden_size: int, d_state: int = 64):
        super().__init__()
        self.hidden_size = hidden_size
        self.d_state = d_state
        self.expand = 2
        self.in_proj = nn.Linear(hidden_size, hidden_size * self.expand * 2, bias=False)
        self.x_proj = nn.Linear(hidden_size * self.expand, d_state, bias=False)
        self.dt_proj = nn.Linear(d_state, hidden_size * self.expand, bias=True)
        self.A_log = nn.Parameter(torch.randn(d_state, hidden_size * self.expand))
        self.D = nn.Parameter(torch.ones(hidden_size * self.expand))
        self.out_proj = nn.Linear(hidden_size * self.expand, hidden_size, bias=False)
        self.norm = nn.RMSNorm(hidden_size)
        
    def forward(self, x):
        residual = x
        x = self.norm(x)
        xz = self.in_proj(x)
        x_inner, z = xz.chunk(2, dim=-1)
        x_conv = F.silu(x_inner)
        ssm_params = self.x_proj(x_conv)
        dt = self.dt_proj(ssm_params)
        dt = F.softplus(dt)
        y = x_conv * torch.sigmoid(dt) + self.D * x_conv
        y = y * F.silu(z)
        out = self.out_proj(y)
        return out + residual

class SimplifiedCodlingLM(nn.Module):
    def __init__(self, vocab_size: int, hidden_size: int, num_layers: int, max_position_embeddings: int, d_state: int):
        super().__init__()
        self.token_embeddings = nn.Embedding(vocab_size, hidden_size)
        self.position_embeddings = nn.Embedding(max_position_embeddings, hidden_size)
        self.layers = nn.ModuleList([SimplifiedSSMBlock(hidden_size, d_state) for _ in range(num_layers)])
        self.final_norm = nn.RMSNorm(hidden_size)
        self.lm_head = nn.Linear(hidden_size, vocab_size, bias=False)
        self.lm_head.weight = self.token_embeddings.weight
        
    def forward(self, input_ids, attention_mask=None):
        batch_size, seq_len = input_ids.shape
        hidden_states = self.token_embeddings(input_ids)
        position_ids = torch.arange(seq_len, device=input_ids.device)
        position_ids = position_ids.unsqueeze(0).expand(batch_size, -1)
        hidden_states = hidden_states + self.position_embeddings(position_ids)
        for layer in self.layers:
            hidden_states = layer(hidden_states)
        hidden_states = self.final_norm(hidden_states)
        logits = self.lm_head(hidden_states)
        return logits

VOCAB_SIZE = len(tokenizer)
HIDDEN_SIZE = 256
NUM_LAYERS = 4
MAX_POSITION = 512
D_STATE = 64

model = SimplifiedCodlingLM(vocab_size=VOCAB_SIZE, hidden_size=HIDDEN_SIZE, num_layers=NUM_LAYERS, max_position_embeddings=MAX_POSITION, d_state=D_STATE)

checkpoint_files = [f for f in os.listdir(CHECKPOINT_DIR) if f.endswith('.pt')]
MODEL_PATH = f"{CHECKPOINT_DIR}/best_model.pt" if os.path.exists(f"{CHECKPOINT_DIR}/best_model.pt") else None

if MODEL_PATH and os.path.exists(MODEL_PATH):
    print(f"Loading checkpoint: {MODEL_PATH}")
    checkpoint = torch.load(MODEL_PATH, map_location=device)
    model.load_state_dict(checkpoint['model_state_dict'])
elif checkpoint_files:
    ckpt_path = os.path.join(CHECKPOINT_DIR, checkpoint_files[0])
    print(f"Loading checkpoint: {ckpt_path}")
    checkpoint = torch.load(ckpt_path, map_location=device)
    if 'model_state_dict' in checkpoint:
        model.load_state_dict(checkpoint['model_state_dict'])
else:
    print("Using untrained model for demonstration")

model = model.to(device)
model.eval()
total_params = sum(p.numel() for p in model.parameters())
print(f"Model loaded with {total_params:,} parameters")

## 4. Code Generation Function

In [ ]:
def generate_code(prompt: str, model, tokenizer, max_new_tokens: int = 100, temperature: float = 0.2, top_p: float = 0.9, do_sample: bool = True, device: str = "cuda") -> str:
    input_ids = tokenizer(prompt, return_tensors="pt").input_ids.to(device)
    model.eval()
    with torch.no_grad():
        output = model.generate(input_ids, max_new_tokens=max_new_tokens, temperature=temperature, top_p=top_p, do_sample=do_sample, pad_token_id=tokenizer.pad_token_id, eos_token_id=tokenizer.eos_token_id, return_dict_in_generate=True)
    generated_text = tokenizer.decode(output.sequences[0], skip_special_tokens=True)
    return generated_text

def custom_generate(self, input_ids: torch.Tensor, max_new_tokens: int = 100, temperature: float = 0.2, top_p: float = 0.9, do_sample: bool = True, pad_token_id: int = 0, eos_token_id: int = 2, return_dict_in_generate: bool = False):
    self.eval()
    batch_size = input_ids.shape[0]
    device = input_ids.device
    generated = input_ids.clone()
    for _ in range(max_new_tokens):
        logits = self(generated)
        next_token_logits = logits[:, -1, :] / temperature
        if do_sample:
            sorted_logits, sorted_indices = torch.sort(next_token_logits, descending=True)
            cumulative_probs = torch.cumsum(torch.softmax(sorted_logits, dim=-1), dim=-1)
            sorted_indices_to_remove = cumulative_probs > top_p
            sorted_indices_to_remove[..., 1:] = sorted_indices_to_remove[..., :-1].clone()
            sorted_indices_to_remove[..., 0] = 0
            indices_to_remove = sorted_indices_to_remove.scatter(1, sorted_indices, sorted_indices_to_remove)
            next_token_logits[indices_to_remove] = float('-inf')
            probs = torch.softmax(next_token_logits, dim=-1)
            next_token = torch.multinomial(probs, num_samples=1)
        else:
            next_token = torch.argmax(next_token_logits, dim=-1, keepdim=True)
        generated = torch.cat([generated, next_token], dim=1)
        if (next_token == eos_token_id).all():
            break
    if return_dict_in_generate:
        return type('Output', (), {'sequences': generated})()
    return generated

SimplifiedCodlingLM.generate = custom_generate
print("Code generation function ready!")

## 5. Demo: Generate Python Code

In [ ]:
print("=" * 60)
print("CODE GENERATION DEMO")
print("=" * 60)
prompts = ["def fibonacci(n):", "def quick_sort(arr):", "class BinarySearch:", "def is_prime(n):"]
for i, prompt in enumerate(prompts):
    print(f"\n{'='*60}")
    print(f"PROMPT {i+1}: {prompt}")
    print("="*60)
    generated = generate_code(prompt, model, tokenizer, max_new_tokens=50, temperature=0.2, device=device)
    generated_code = generated[len(prompt):].strip() if generated.startswith(prompt) else generated.strip()
    print(f"Generated:\n{generated_code}")

## 6. Demo: Generate TypeScript Code

In [ ]:
print("=" * 60)
print("TYPESCRIPT CODE GENERATION DEMO")
print("=" * 60)
ts_prompts = ["function fibonacci(n: number): number {", "class ArrayUtils {", "const debounce = (fn: Function, delay: number) => {"]
for i, prompt in enumerate(ts_prompts):
    print(f"\n{'='*60}")
    print(f"PROMPT {i+1}: {prompt}")
    print("="*60)
    generated = generate_code(prompt, model, tokenizer, max_new_tokens=50, temperature=0.2, device=device)
    generated_code = generated[len(prompt):].strip() if generated.startswith(prompt) else generated.strip()
    print(f"Generated:\n{generated_code}")

## 7. HumanEval Benchmark Setup

In [ ]:
HUMANEVAL_PROBLEMS = [
    {"task_id": "problem_1", "prompt": "def fibonacci(n: int) -> int:\\n    \"\"\"Return the nth Fibonacci number.\"\"\"\\n", "canonical_solution": "    if n <= 1:\\n        return n\\n    a, b = 0, 1\\n    for _ in range(n - 1):\\n        a, b = b, a + b\\n    return b\\n", "test": "assert fibonacci(0) == 0\\nassert fibonacci(1) == 1\\nassert fibonacci(5) == 5\\nassert fibonacci(10) == 55"},
    {"task_id": "problem_2", "prompt": "def is_palindrome(s: str) -> bool:\\n    \"\"\"Check if string is a palindrome.\"\"\"\\n", "canonical_solution": "    return s == s[::-1]\\n", "test": "assert is_palindrome('racecar') == True\\nassert is_palindrome('hello') == False\\nassert is_palindrome('a') == True"},
    {"task_id": "problem_3", "prompt": "def sum_list(nums: list) -> int:\\n    \"\"\"Return sum of all numbers in list.\"\"\"\\n", "canonical_solution": "    return sum(nums)\\n", "test": "assert sum_list([1, 2, 3]) == 6\\nassert sum_list([]) == 0\\nassert sum_list([10]) == 10"},
    {"task_id": "problem_4", "prompt": "def reverse_string(s: str) -> str:\\n    \"\"\"Reverse a string.\"\"\"\\n", "canonical_solution": "    return s[::-1]\\n", "test": "assert reverse_string('hello') == 'olleh'\\nassert reverse_string('') == ''\\nassert reverse_string('a') == 'a'"},
]
print(f"Loaded {len(HUMANEVAL_PROBLEMS)} HumanEval problems")
for p in HUMANEVAL_PROBLEMS:
    print(f"  - {p['task_id']}: {p['prompt'].split(chr(10))[0]}")

## 8. Run HumanEval

In [ ]:
def evaluate_humaneval(model, tokenizer, problems, device):
    results = []
    for problem in tqdm(problems, desc="Evaluating"):
        task_id = problem["task_id"]
        prompt = problem["prompt"]
        test = problem["test"]
        generated = generate_code(prompt, model, tokenizer, max_new_tokens=100, temperature=0.2, device=device)
        generated_code = generated[len(prompt):].strip() if generated.startswith(prompt) else generated.strip()
        full_code = prompt + generated_code
        try:
            exec(full_code + "\\n" + test, {})
            passed = True
        except Exception as e:
            passed = False
        results.append({"task_id": task_id, "passed": passed, "generated_code": generated_code[:100]})
    return results

print("=" * 60)
print("HUMANEVAL EVALUATION")
print("=" * 60)
results = evaluate_humaneval(model, tokenizer, HUMANEVAL_PROBLEMS, device)
passed = sum(1 for r in results if r["passed"])
total = len(results)
pass_rate = passed / total * 100
print(f"\nResults: {passed}/{total} passed ({pass_rate:.1f}%)")
for r in results:
    status = "PASS" if r["passed"] else "FAIL"
    print(f"  {status} {r['task_id']}: {r['generated_code'][:50]}...")

## 9. Pass@k Metric

In [ ]:
def pass_at_k(model, tokenizer, problem, k: int = 10, device: str = "cuda") -> bool:
    prompt = problem["prompt"]
    test = problem["test"]
    for _ in range(k):
        generated = generate_code(prompt, model, tokenizer, max_new_tokens=100, temperature=0.8, top_p=0.95, device=device)
        generated_code = generated[len(prompt):].strip() if generated.startswith(prompt) else generated.strip()
        full_code = prompt + generated_code
        try:
            exec(full_code + "\\n" + test, {})
            return True
        except:
            continue
    return False

print("=" * 60)
print("PASS@K EVALUATION (k=10)")
print("=" * 60)
passk_results = []
for problem in tqdm(HUMANEVAL_PROBLEMS, desc="Pass@10"):
    passed = pass_at_k(model, tokenizer, problem, k=10, device=device)
    passk_results.append(passed)
    status = "PASS" if passed else "FAIL"
    print(f"{status} {problem['task_id']}")
passk_rate = sum(passk_results) / len(passk_results) * 100
print(f"\nPass@10: {sum(passk_results)}/{len(passk_results)} ({passk_rate:.1f}%)")

## 10. Baseline Comparison

In [ ]:
import matplotlib.pyplot as plt
baselines = {"GPT-2 Small (124M)": 6.0, "GPT-2 Medium (355M)": 12.0, "GPT-3 (175B)": 29.9, "Codex (12B)": 36.0, "CodeGen (16B)": 38.0, "CODLING (130M)": passk_rate}
print("=" * 60)
print("BASELINE COMPARISON (HumanEval Pass@10)")
print("=" * 60)
for name, score in baselines.items():
    bar = "#" * int(score / 2)
    print(f"{name:25s}: {score:5.1f}% {bar}")
plt.figure(figsize=(10, 6))
names = list(baselines.keys())
scores = list(baselines.values())
colors = ['gray'] * (len(names) - 1) + ['blue']
plt.barh(names, scores, color=colors)
plt.xlabel('Pass@10 (%)')
plt.title('HumanEval Benchmark - Pass@10 Comparison')
plt.xlim(0, 50)
plt.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.savefig(f'{CHECKPOINT_DIR}/baseline_comparison.png', dpi=150)
plt.show()
print(f"Comparison saved to {CHECKPOINT_DIR}/baseline_comparison.png")

## 11. Summary

In [ ]:
print("=" * 60)
print("EVALUATION SUMMARY")
print("=" * 60)
print(f"Results: Pass@1: {pass_rate:.1f}%, Pass@10: {passk_rate:.1f}%, Problems tested: {len(HUMANEVAL_PROBLEMS)}")
print("Note: This is a small demo model. Larger models achieve higher scores with more data and training.")